# KD: BEATs fine-tune → CED-Tiny (`kd-beat-ced`)

Notebook này triển khai pipeline chưng cất tri thức trong `imp/kd-beat-ced`. Teacher là **BEATs đã fine-tune trên fold 0** (đóng băng), student là **CED-Tiny** (DeiT-Tiny khởi tạo từ AudioSet). Teacher và student nhìn cùng một đầu vào n̂, là phần nhiễu FRCRN đã tách sẵn và lưu trong cache.

Notebook chỉ gọi lớp và hàm trong `src/`, `scripts/` và `main.py`. Vòng huấn luyện nằm trong `KDTrainer`, không viết lại trong cell.

Mặc định `RUN_TRAINING = False` chạy smoke test: 8 mẫu tổng hợp, 1 epoch, ghi vào `output/test/`. Đặt `True` để huấn luyện thật khi đã có cache n̂.

Sơ đồ tổng quan:

![Pipeline KD](figures/kd-beat-ced/pipeline_overview.svg)

## Kiến trúc

**Teacher: BEATs fold 0.** BEATs base có 12 khối Transformer, 768 chiều, 12 head. Đầu vào là fbank 128 mel (khung 25 ms, bước 10 ms) và patch 16 × 16, cho ra 24 × 8 = 192 token trên 4 giây. BEATs không có token CLS; đầu ra lấy trung bình các token. Sau khi nạp checkpoint fold, `load_fold_teacher` đóng băng toàn bộ tham số.

**Student: CED-Tiny.** Log-mel 64 mel, 398 khung, có SpecAugment khi huấn luyện. Patch 16 × 16 cho 4 × 24 = 96 token, thêm một token CLS. Backbone là DeiT-Tiny: 12 khối, 192 chiều, 3 head. Khác DeiT thường, mỗi khối attention có bias vị trí tương đối và cổng kiểu GRU (`ViTSBias`), giống BEATs. Trọng số khởi tạo từ CED-Tiny AudioSet.

Hai mô hình khác số token (192 và 96) và số head (12 và 3), nên không so trực tiếp được. Trước khi so attention, teacher được gộp về lưới của student.

| | Teacher BEATs | Student CED-Tiny |
|---|---|---|
| Đầu vào | n̂ từ cache, RMS 0,07925 | n̂ từ cache, RMS 0,07925 |
| Biểu diễn | fbank 128 mel | log-mel 64 mel, 398 khung |
| Token | 192 = 24 × 8 | 96 = 24 × 4, cộng CLS |
| Head × khối | 12 × 12 | 3 × 12 |
| Trọng số | fine-tune fold 0, đóng băng | CED-Tiny, huấn luyện |

Ánh xạ token và head trước khi so sánh:

![Ánh xạ token](figures/kd-beat-ced/token_mapping.svg)

## Hàm mất mát

`KDTrainer.training_step` cộng bốn thành phần, cùng AT và bias khi chúng được bật trong mục `attention`:

$$L = \mathrm{CE}_{\text{LS}}(y, p_s) + w_{KD}\,T^2\,\mathrm{KL}(p_t \,\|\, p_s) + \beta\,L_{AT} + w_{\text{scale}}\,L_{\text{scale}} + w_{\text{pos}}\,L_{\text{pos}}$$

- **CE** với label smoothing 0,1, dùng nhãn đã mixup (`SoftTargetCrossEntropy`).
- **Logit KD** (`LogitDistillationLoss`): KL giữa softmax của teacher và student ở T = 4, nhân T².
- **AT** (`src/loss/attention_kd.py`): energy của mỗi token là trung bình bình phương hidden state theo chiều đặc trưng. Teacher gộp tần số 8 → 4, cả hai được chuẩn hoá L2 rồi so MSE ở layer 4, 8, 12. Student bỏ CLS.
- **BEATs bias**: scale của cổng GRU và position bias, lấy từ cache attention của layer 1 đến 12. Teacher gộp head 12 → 3 và gộp tần số như AT; position bias gộp cả hàng và cột.

Mixup trộn n̂ và nhãn với cùng λ trước khi đưa vào cả hai mô hình, nên teacher và student luôn thấy cùng một đầu vào.

Giá trị attention mặc định (β = 1000, weight 1,0 cho từng phần bias) là điểm xuất phát, chưa tune. Khi chạy thật, so độ lớn các số hạng `at`, `bias_scale`, `bias_pos` trong lịch sử (mục 7) với CE trước khi đổi.

## (1) Tên pipeline, chế độ chạy, đầu ra

`RUN_TRAINING = False` ghi vào `output/test/smoke-<thời gian>/`. `True` ghi vào `output/` với tên `kd-ced-tiny-<thời gian>`, cùng tên mà `main.py train` dùng. `RUN_ID` có giá trị thì nối tiếp đúng lần cũ qua `train_state.pt`, không tạo thư mục mới. Logger ghi mốc chính vào `run.log_file`; tiến độ từng batch hiện bằng tqdm.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "rules.MD").is_file())
PROJECT_ROOT = REPO_ROOT / "imp" / "kd-beat-ced"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import torch

from src.utils.logger import setup_logger
from src.utils.paths import RunPaths

PIPELINE_NAME = "kd-beat-ced"
RUN_TRAINING = False  # True: huấn luyện thật vào output/; False: smoke test nhỏ vào output/test/
RUN_ID = None  # ví dụ "kd-ced-tiny-2026-10-29-10-00-00" để nối tiếp một lần đã dừng

OUTPUT_DIR = PROJECT_ROOT / "output" if RUN_TRAINING else PROJECT_ROOT / "output" / "test"
EXPERIMENT = "kd-ced-tiny" if RUN_TRAINING else "smoke"
run = (RunPaths.from_run_id(RUN_ID, OUTPUT_DIR) if RUN_ID else RunPaths.new(EXPERIMENT, OUTPUT_DIR)).create()
logger = setup_logger(PIPELINE_NAME, run.log_file)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{PIPELINE_NAME} | RUN_TRAINING={RUN_TRAINING} | device={device} | run={run.run_id}")

## (2) Tham số cấu hình

Cấu hình gốc ở `config/kd_beat_ced.yaml`, đọc bằng `read_config` và dựng thành dataclass giống `main.py`. Khi `RUN_TRAINING = False`, chỉ số epoch, batch size, số mẫu đánh giá và `amp` bị thu nhỏ; công thức loss và kiến trúc không đổi.

| Khoá trong yaml | Giá trị | Ý nghĩa |
|---|---|---|
| `loss.temperature`, `loss.weight` | 4,0 · 1,0 | T và trọng số của logit KD |
| `loss.label_smoothing` | 0,1 | Label smoothing trong CE |
| `attention.at_weight`, `attention.at_layers` | 1000 · [4, 8, 12] | β và layer của AT |
| `attention.bias_scale_weight`, `attention.bias_pos_weight` | 1,0 · 1,0 | Trọng số scale và position bias |
| `attention.bias_layers` | 1 đến 12 | Layer lấy cache bias |
| `teacher.input_rms` | 0,07925 | RMS chuẩn hoá n̂ cho teacher và student |
| `train.lr`, `train.epochs`, `train.patience` | 1e-4 · 30 · 7 | AdamW, số epoch tối đa, early stop |

Bảng đầy đủ nằm trong `imp/kd-beat-ced/README.md`.

In [ ]:
from dataclasses import replace

from main import CONFIG, read_config
from src.loss.attention_kd import AttentionKDConfig
from src.train import DatasetConfig, KDLossConfig, KDTrainConfig

SMOKE_SAMPLES = 8
NOISE_CACHE = None  # thư mục chứa train.npy, validation.npy và .done; None: lấy dataset.noise_cache trong yaml

raw = read_config(CONFIG)
dataset_config = DatasetConfig.from_dict(raw["dataset"])
train_config = KDTrainConfig.from_dict(raw["train"])
loss_config = KDLossConfig.from_dict(raw["loss"])
attention_config = AttentionKDConfig.from_dict(raw.get("attention", {}))
if not RUN_TRAINING:
    train_config = replace(
        train_config,
        epochs=1,
        patience=2,
        batch_size=2,
        eval_batch_size=2,
        warmup_steps=1,
        train_eval_clips=4,
        num_workers=0,
        amp=False,
    )
print("loss", loss_config)
print("attention", attention_config)
print("train", train_config)

## (3) Dataset

- **Smoke** (`RUN_TRAINING = False`): `SyntheticDataset` gồm 8 mẫu nhãn ngẫu nhiên, chỉ để kiểm tra đường đi của dữ liệu và loss.
- **Thật** (`RUN_TRAINING = True`): `NoiseCacheDataset` đọc n̂ từ `train.npy` và `validation.npy`. Mỗi file có `.done` ghi số mẫu, phải khớp manifest. Thứ tự cache phải đúng thứ tự manifest; notebook không sắp lại.

Notebook không nạp tập test. Đánh giá test chạy bằng `main.py eval --split test`.

In [ ]:
from main import resolved_cache
from scripts.load_dataset import SyntheticDataset, build_cached_loader, build_cached_loaders, build_dataloader

if RUN_TRAINING:
    cache = resolved_cache(raw, NOISE_CACHE)
    loaders = build_cached_loaders(
        dataset_config,
        cache,
        train_config.batch_size,
        train_config.eval_batch_size,
        None,
        train_config.num_workers,
    )
    train_loader, val_loader = loaders.train, loaders.validation
    class_names = loaders.class_names
    train_eval_loader = (
        build_cached_loader(
            dataset_config,
            "train",
            cache,
            train_config.eval_batch_size,
            train_config.train_eval_clips,
            train_config.num_workers,
            shuffle=False,
        )
        if train_config.train_eval_clips > 0
        else None
    )
else:
    dataset = SyntheticDataset(length=SMOKE_SAMPLES, num_classes=36)
    train_loader = build_dataloader(dataset, train_config.batch_size, shuffle=True, num_workers=0)
    val_loader = build_dataloader(dataset, train_config.eval_batch_size, shuffle=False, num_workers=0)
    train_eval_loader = val_loader
    class_names = [f"class_{index}" for index in range(36)]
print(f"train batches {len(train_loader)} | validation batches {len(val_loader)} | classes {len(class_names)}")

## (4) Teacher BEATs fold 0 (đóng băng)

`load_fold_teacher` đọc các khoá `encoder.model.*` của checkpoint fold, ghép với `head.weight` và `head.bias` thành `BEATClassifier`, kiểm tra không thiếu hay thừa khoá rồi đóng băng. Hàm báo lỗi nếu checkpoint không đúng định dạng fold. Teacher được đưa sang `device` và luôn ở chế độ eval.

In [ ]:
from main import resolve_path
from src.model.beat import load_fold_teacher

teacher, teacher_meta = load_fold_teacher(
    resolve_path(raw["teacher"]["checkpoint"]), resolve_path(raw["teacher"]["config"]), device
)
teacher_total = sum(parameter.numel() for parameter in teacher.parameters())
teacher_trainable = sum(parameter.numel() for parameter in teacher.parameters() if parameter.requires_grad)
print(f"teacher {teacher_meta} | {teacher_total / 1e6:.2f}M tham số | trainable {teacher_trainable}")

## (5) Student CED-Tiny và kiểm tra shape

`build_student` của `main.py` dựng student từ cùng yaml. Ở smoke, `init = none` để không tải trọng số từ Hugging Face. Khi huấn luyện thật, `init = ced-tiny`: nạp từ `student.ced_path` nếu có, nếu không thì tải từ Hugging Face.

Ô kiểm tra shape chạy một batch qua teacher và student, rồi in shape token trước và sau gộp, để đối chiếu với bảng ở mục kiến trúc.

In [ ]:
from main import build_student

overrides = {} if RUN_TRAINING else {"init": "none"}
student = build_student(raw, overrides).to(device)
print(f"student init {student.config.init} | {student.init_report}")
for name, count in student.describe_params().items():
    print(f"  {name:36s} {count:>12,d}")
print(f"tổng {student.num_parameters() / 1e6:.2f}M tham số")

In [ ]:
from src.loss.attention_kd import pool_frequency, token_energy
from src.utils.files import rms_normalize

student.eval()
batch = next(iter(train_loader))
with torch.no_grad():
    noise = rms_normalize(batch["waveform"].to(device), float(raw["teacher"]["input_rms"]))
    teacher_out = teacher(noise, layers=attention_config.at_layers)
    student_out = student.forward_with_hidden(noise, attention_config.at_layers)
print(f"logits: teacher {tuple(teacher_out.logits.shape)} | student {tuple(student_out.logits.shape)}")
for layer, t_hidden, s_hidden in zip(
    attention_config.at_layers, teacher_out.hidden, student_out.hidden, strict=True
):
    t_map = pool_frequency(
        token_energy(t_hidden),
        1,
        attention_config.time_patches,
        attention_config.teacher_freq_patches,
        attention_config.student_freq_patches,
    )
    s_map = token_energy(s_hidden)
    print(
        f"layer {layer:2d}: teacher hidden {tuple(t_hidden.shape)} → energy sau gộp {tuple(t_map.shape)}"
        f" | student hidden {tuple(s_hidden.shape)} → energy {tuple(s_map.shape)}"
    )

## (6) Huấn luyện và đánh giá

`KDTrainer.fit` chạy vòng lặp epoch, early stop, lưu checkpoint tốt nhất và `train_state.pt`. Mỗi batch:

1. Mixup trộn n̂ và nhãn, rồi đưa vào teacher (`no_grad`) và student cùng lúc.
2. Tính CE, logit KD, AT và bias theo công thức ở phần kiến trúc.

Sau mỗi epoch, validation tính `score = macro_acc − 0,2 × gap`, với `gap` là chênh lệch giữa accuracy trên tập train_eval và top-1 trên validation. Nếu score tốt hơn, checkpoint được ghi đè. Nếu không cải thiện, lr giảm theo `lr_decay` khi đạt `lr_patience`.

Khi fit xong, notebook đánh giá lại validation và ghi `result.csv` theo từng mức SNR.

In [ ]:
from main import result_rows, save_history, write_result_csv
from src.train import KDTrainer

trainer = KDTrainer(
    student,
    teacher,
    train_config,
    loss_config,
    device,
    logger,
    PIPELINE_NAME,
    run.checkpoint_path("student"),
    run.train_state_path("student"),
    float(raw["teacher"]["input_rms"]),
    train_eval_loader=train_eval_loader,
    resume=RUN_ID is not None,
    attention=attention_config,
)
result = trainer.fit(train_loader, val_loader)
metrics = trainer.evaluate(val_loader, desc="validation")
write_result_csv(run.result_file, result_rows(metrics, "validation"))
save_history(
    run.history_path("student"),
    {"run_id": run.run_id, "result": result.to_dict(), "validation": metrics, "config": raw},
)
print(
    f"best epoch {result.best_epoch} | macro_acc {metrics['macro_acc']:.4f}"
    f" | score {metrics['score']:.4f} | gap {metrics['gap']:.4f}"
)

## (7) Kết quả và biểu đồ

Biểu đồ trái: các số hạng loss trung bình theo epoch. Dùng để so độ lớn `at`, `bias_scale`, `bias_pos` với `train_loss` trước khi giữ β và trọng số bias. Biểu đồ phải: validation theo epoch. Bảng dưới là `result.csv` theo từng mức SNR.

In [ ]:
import csv

import matplotlib.pyplot as plt

history = result.history
epochs = [row["epoch"] for row in history]
fig, (ax_loss, ax_val) = plt.subplots(1, 2, figsize=(12, 4))
for key in ("train_loss", "kd", "at", "bias_scale", "bias_pos"):
    ax_loss.plot(epochs, [row.get(key, 0.0) for row in history], marker="o", label=key)
ax_loss.set_xlabel("epoch")
ax_loss.set_title("Số hạng loss trung bình")
ax_loss.legend()
for key in ("val_score", "val_macro_acc", "val_mAP_macro"):
    ax_val.plot(epochs, [row.get(key, float("nan")) for row in history], marker="o", label=key)
ax_val.set_xlabel("epoch")
ax_val.set_title("Validation")
ax_val.legend()
fig.tight_layout()
fig.savefig(run.figure_dir / "history.png", dpi=120)
plt.show()


def fmt(value):
    return f"{float(value):.4f}" if value not in (None, "") else "-"


with open(run.result_file, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(
            f"SNR {row['snr_db']:>8s} | top1 {fmt(row['top1_accuracy'])} | balanced {fmt(row['balanced_accuracy'])}"
            f" | mAP {fmt(row['mAP'])} | macro_f1 {fmt(row['macro_f1'])}"
        )
print(f"score tổng {fmt(metrics['score'])} | macro_acc tổng {fmt(metrics['macro_acc'])} | gap {fmt(metrics['gap'])}")

## Lệnh tương đương ngoài notebook

Cùng pipeline chạy được từ dòng lệnh trong `imp/kd-beat-ced`:

```bash
python main.py smoke
python main.py train --noise-cache <thư-mục-cache>
python main.py train --noise-cache <thư-mục-cache> --run-id kd-ced-tiny-2026-10-29-10-00-00
python main.py eval --checkpoint output/checkpoint/<run-id>/student/last_best_checkpoint.pt --noise-cache <thư-mục-cache> --split test
```

Lệnh `eval --split test` chỉ chạy sau khi chốt cấu hình, để tập test không tham gia chọn siêu tham số.